In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Step 1 - Load the data

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/My Drive/volleypred/'
    print("Running in Colab. Drive mounted.")
except ImportError:
    BASE_PATH = '../'
    print("Running locally. Using relative paths.")

DATA_PATH = os.path.join(BASE_PATH, 'data/Mens-Volleyball-PlusLiga-2008-2023.csv')
df = pd.read_csv(DATA_PATH)

### Basic information

In [ ]:
df.columns
df.info()

The dataset consist of 2639 rows (so 2639 matches from 14 seasons) and 44 columns (date, team 1, team 2, score in 2 columns, winner + 19 statistics for each team).

Many columns are in the wrong format, so converting them to proper formats is necessary.

### Checking the number of teams

In [ ]:
df['Team_1'].unique()
df['Team_2'].unique()

len(df['Team_1'].unique())
len(df['Team_2'].unique())

There are 23 unique teams in the dataset.

### Convert the column types

In [ ]:
df_preprocessed = df.copy()
df_preprocessed['Date'] = pd.to_datetime(df_preprocessed['Date'], format='%d.%m.%Y, %H:%M')

In [ ]:
object_cols = df_preprocessed.select_dtypes(include='object').columns
object_cols

cols_to_convert = object_cols.drop(['Team_1', 'Team_2'])
cols_to_convert

df_preprocessed[cols_to_convert].head()

In [ ]:
perc_cols = ['T1_Srv_Eff', 'T1_Rec_Pos', 'T1_Rec_Perf', 'T1_Att_Kill_Perc', 'T1_Att_Eff', 'T1_Att_Sum',
             'T2_Srv_Eff', 'T2_Rec_Pos', 'T2_Rec_Perf', 'T2_Att_Kill_Perc', 'T2_Att_Eff', 'T2_Att_Sum']

for col in perc_cols:
    df_preprocessed[col] = pd.to_numeric(df_preprocessed[col].str.replace('%', ''))

perc_cols.remove('T1_Att_Sum')
perc_cols.remove('T2_Att_Sum')

for col in perc_cols:
    df_preprocessed[col] = df_preprocessed[col] / 100

df_preprocessed.head()

In [ ]:
perc_cols = ['T1_Srv_Eff', 'T1_Rec_Pos', 'T1_Rec_Perf', 'T1_Att_Kill_Perc', 'T1_Att_Eff', 'T1_Att_Sum',
             'T2_Srv_Eff', 'T2_Rec_Pos', 'T2_Rec_Perf', 'T2_Att_Kill_Perc', 'T2_Att_Eff', 'T2_Att_Sum']

float_cols = cols_to_convert.drop(perc_cols)

for col in float_cols:
    df_preprocessed[col] = pd.to_numeric(df_preprocessed[col].str.replace(',', '.'))


In [ ]:
df_preprocessed.select_dtypes(include='object').columns

Only "Team_1" and "Team_2" columns remained as the object type columns.

In [ ]:
df_preprocessed.head()
df_preprocessed.info()

In [ ]:
df_preprocessed.columns[df_preprocessed.isna().any()]
df_preprocessed.shape

#Step 2 - Add new columns

In [ ]:
def get_season(date):
    year = date.year
    if date.month >= 9:
        return f"{year}/{year + 1}"
    else:
        return f"{year - 1}/{year}"

df_preprocessed['Season'] = df_preprocessed['Date'].apply(get_season)
df_preprocessed.tail()

In [ ]:
df_preprocessed['Season'].unique()
df_preprocessed['Season'].isna().sum()

In [ ]:
df_preprocessed['Score'] = df_preprocessed['T1_Score'].astype(int).astype(str) + ':' + df_preprocessed['T2_Score'].astype(int).astype(str)
df_preprocessed.head()

In [ ]:
df_preprocessed['Score'].isna().sum()

In [ ]:
df_preprocessed = df_preprocessed.sort_values(by='Date').reset_index(drop=True)
df_preprocessed.head()

# Step 3 - Team-level format

### Divide the table by teams

In [ ]:
df_preprocessed.columns

base_stats = ['Sum', 'BP', 'Ratio', 'Srv_Sum', 'Srv_Err', 'Srv_Ace',
              'Srv_Eff', 'Rec_Sum', 'Rec_Err', 'Rec_Pos', 'Rec_Perf',
              'Att_Sum', 'Att_Err', 'Att_Blk', 'Att_Kill',
              'Att_Kill_Perc', 'Att_Eff', 'Blk_Sum', 'Blk_As',]

In [ ]:
t1_stats = ['T1_' + s for s in base_stats]
t2_stats = ['T2_' + s for s in base_stats]

t1_df = df_preprocessed[['Date', 'Season', 'Team_1'] + t1_stats].copy()
t1_df.columns = ['Date', 'Season', 'Team'] + base_stats
t1_df['Match_ID'] = t1_df.index
t1_df['Side'] = 'Team_1'

t2_df = df_preprocessed[['Date', 'Season', 'Team_2'] + t2_stats].copy()
t2_df.columns = ['Date', 'Season', 'Team'] + base_stats
t2_df['Match_ID'] = t2_df.index
t2_df['Side'] = 'Team_2'

In [ ]:
team_level_df = pd.concat([t1_df, t2_df], ignore_index=True)
team_level_df = team_level_df.sort_values(['Team', 'Season', 'Date'])

### Identify first matches for each team in every season and flague it

In [ ]:
team_level_df['is_first_match_season'] = (
    team_level_df.groupby(['Team', 'Season']).cumcount() == 0
)

len(team_level_df[team_level_df['is_first_match_season'] == True])
team_level_df.head()

### Rolling averages

In [ ]:
rolling_cols = [f'Rolling_{col}' for col in base_stats]

team_level_df[rolling_cols] = (
    team_level_df
    .groupby('Team')[base_stats]
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

### Days since last match

In [ ]:
team_level_df['Days_since_last_match'] = (
    team_level_df.groupby('Team')['Date'].diff().dt.days
)

### Drop the first match

In [ ]:
matches_to_drop = team_level_df.loc[
    team_level_df['is_first_match_season'],
    'Match_ID'
].unique()

filtered_teams_df = team_level_df[
    ~team_level_df['Match_ID'].isin(matches_to_drop)
].copy()

filtered_teams_df.head()

# Step 4 - Match-level format

### Merge

In [ ]:
t1_side = filtered_teams_df[filtered_teams_df['Side'] == 'Team_1']
t2_side = filtered_teams_df[filtered_teams_df['Side'] == 'Team_2']

In [ ]:
match_df = t1_side.merge(
    t2_side,
    on=['Match_ID', 'Date', 'Season'],
    suffixes=('_T1', '_T2')
)
match_df.shape

### Difference-based statistics

In [ ]:
for stat in base_stats:
    match_df[f'Diff_{stat}'] = (
        match_df[f'Rolling_{stat}_T1'] -
        match_df[f'Rolling_{stat}_T2']
    )

match_df['Diff_Days_since_last_match'] = (
    match_df['Days_since_last_match_T1'] -
    match_df['Days_since_last_match_T2']
)

match_df.head()

### Final dataset reduction

In [ ]:
final_cols = (
    ['Match_ID', 'Date', 'Season', 'Team_T1', 'Team_T2'] +
    [c for c in match_df.columns if c.startswith('Diff_')]
)
final_df = match_df[final_cols].copy()

final_df = final_df.merge(
    df_preprocessed[['Score']],
    left_on = 'Match_ID',
    right_index=True
)

final_df.drop(columns=['Match_ID'], inplace=True)
final_df.head()

### Validation

In [ ]:
assert final_df.isna().sum().sum() == 0
assert final_df['Score'].nunique() == 6

#

# Step 5

In [ ]:
final_df.sort_values(by=['Date', 'Season'], inplace=True)
final_df.reset_index(drop=True, inplace=True)
final_df.head()

In [ ]:
FINAL_DATA_OUTPUT_PATH = os.path.join(BASE_PATH, 'data/final_df.csv')

final_df.to_csv(FINAL_DATA_OUTPUT_PATH, index=False)

print(f"The dataset has been saved to: {FINAL_DATA_OUTPUT_PATH}")